In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import joblib
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------
# Paths & config
# --------------------
train_dir = r"E:\Road_Quality\Frame_Num\Unmatch"
save_dir = r"E:\Road_Quality\model\CSI_percentile_linear"
os.makedirs(save_dir, exist_ok=True)

seed = 42
MAX_SAMPLES = 20000

# --------------------
# Utilities
# --------------------
def list_jpgs(folder):
    return [
        os.path.join(folder, x)
        for x in os.listdir(folder)
        if x.lower().endswith(".jpg")
    ]

def compute_csi(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.std(lap))

def compute_lcr(gray):
    edges = cv2.Canny(gray, 100, 200)
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=50,
        minLineLength=30,
        maxLineGap=10,
    )
    if lines is None:
        return 0.0

    tot = 0.0
    for l in lines:
        x1, y1, x2, y2 = l[0]
        tot += float(np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2))

    return float(tot / (edges.size + 1e-6))

def percentile01(vals):
    a = np.asarray(vals, dtype=np.float32)
    n = len(a)
    if n <= 1:
        return np.zeros_like(a)

    order = np.argsort(a)
    ranks = np.empty(n, dtype=np.float32)
    ranks[order] = np.arange(n, dtype=np.float32)
    return ranks / (n - 1)

# --------------------
# Load & subsample paths
# --------------------
paths = list_jpgs(train_dir)

rng = np.random.default_rng(seed)
if len(paths) > MAX_SAMPLES:
    paths = rng.choice(paths, size=MAX_SAMPLES, replace=False).tolist()

print(f"Using {len(paths)} images for training")

# --------------------
# Feature extraction
# --------------------
X = []
y_raw = []

for p in tqdm(paths, desc="Extracting features (Unmatch – 20k subset)"):
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        continue

    csi = compute_csi(g)
    lcr = compute_lcr(g)
    mu = float(np.mean(g))
    sd = float(np.std(g))

    X.append([csi, lcr, mu, sd])
    y_raw.append(csi)

X = np.asarray(X, dtype=np.float32)
y_raw = np.asarray(y_raw, dtype=np.float32)

# --------------------
# Percentile target
# --------------------
y = percentile01(y_raw).astype(np.float32)

# --------------------
# Train / test split
# --------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.10, random_state=seed
)

# --------------------
# Linear model (Ridge)
# --------------------
model_csi = Ridge(alpha=1.0, random_state=seed)
model_csi.fit(X_tr, y_tr)

pred = model_csi.predict(X_te)
pred = np.clip(pred, 0.0, 1.0)

mae = mean_absolute_error(y_te, pred)
mse = mean_squared_error(y_te, pred)
r2 = r2_score(y_te, pred)

print("\nCSI-percentile Linear Model (Ridge)")
print("MAE:", mae)
print("MSE:", mse)
print("R2 :", r2)

# --------------------
# Save artifacts
# --------------------
joblib.dump(model_csi, os.path.join(save_dir, "model.joblib"))
np.save(os.path.join(save_dir, "feature_mean.npy"), X.mean(axis=0))

print("Saved to:", save_dir)


Using 20000 images for training


Extracting features (Unmatch – 20k subset): 100%|████████████████████████████████| 20000/20000 [19:47<00:00, 16.84it/s]


CSI-percentile Linear Model (Ridge)
MAE: 0.039067388
MSE: 0.0021433423
R2 : 0.974251005270028
Saved to: E:\Road_Quality\model\CSI_percentile_linear


In [3]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import joblib
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --------------------
# Paths & config
# --------------------
train_dir = r"E:\Road_Quality\Frame_Num\Unmatch"
save_dir = r"E:\Road_Quality\model\LCR_percentile_linear"
os.makedirs(save_dir, exist_ok=True)

seed = 42
MAX_SAMPLES = 20000

# --------------------
# Utilities
# --------------------
def list_jpgs(folder):
    return [
        os.path.join(folder, x)
        for x in os.listdir(folder)
        if x.lower().endswith(".jpg")
    ]

def compute_csi(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(np.std(lap))

def compute_lcr(gray):
    edges = cv2.Canny(gray, 100, 200)
    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=50,
        minLineLength=30,
        maxLineGap=10,
    )
    if lines is None:
        return 0.0

    tot = 0.0
    for l in lines:
        x1, y1, x2, y2 = l[0]
        tot += float(np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2))

    return float(tot / (edges.size + 1e-6))

def percentile01(vals):
    a = np.asarray(vals, dtype=np.float32)
    n = len(a)
    if n <= 1:
        return np.zeros_like(a)

    order = np.argsort(a)
    ranks = np.empty(n, dtype=np.float32)
    ranks[order] = np.arange(n, dtype=np.float32)
    return ranks / (n - 1)

# --------------------
# Load & subsample paths
# --------------------
paths = list_jpgs(train_dir)

rng = np.random.default_rng(seed)
if len(paths) > MAX_SAMPLES:
    paths = rng.choice(paths, size=MAX_SAMPLES, replace=False).tolist()

print(f"Using {len(paths)} images for training")

# --------------------
# Feature extraction
# --------------------
X = []
y_raw = []

for p in tqdm(paths, desc="Extracting features (Unmatch – 20k subset)"):
    g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if g is None:
        continue

    csi = compute_csi(g)
    lcr = compute_lcr(g)
    mu = float(np.mean(g))
    sd = float(np.std(g))

    # Features
    X.append([csi, lcr, mu, sd])
    # Target raw metric = LCR
    y_raw.append(lcr)

X = np.asarray(X, dtype=np.float32)
y_raw = np.asarray(y_raw, dtype=np.float32)

# --------------------
# Percentile target
# --------------------
y = percentile01(y_raw).astype(np.float32)

# --------------------
# Train / test split
# --------------------
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.10, random_state=seed
)

# --------------------
# Linear model (Ridge)
# --------------------
model_lcr = Ridge(alpha=1.0, random_state=seed)
model_lcr.fit(X_tr, y_tr)

pred = model_lcr.predict(X_te)
pred = np.clip(pred, 0.0, 1.0)

mae = mean_absolute_error(y_te, pred)
mse = mean_squared_error(y_te, pred)
r2 = r2_score(y_te, pred)

print("\nLCR-percentile Linear Model (Ridge)")
print("MAE:", mae)
print("MSE:", mse)
print("R2 :", r2)

# --------------------
# Save artifacts
# --------------------
joblib.dump(model_lcr, os.path.join(save_dir, "model.joblib"))
np.save(os.path.join(save_dir, "feature_mean.npy"), X.mean(axis=0))

print("Saved to:", save_dir)


Using 20000 images for training


Extracting features (Unmatch – 20k subset): 100%|████████████████████████████████| 20000/20000 [19:14<00:00, 17.32it/s]


LCR-percentile Linear Model (Ridge)
MAE: 0.093379796
MSE: 0.013447351
R2 : 0.8372433777937512
Saved to: E:\Road_Quality\model\LCR_percentile_linear
